# dip-fmm and regular MagTense demagnetisation

This notebook evolves the same micromagnetic problem for 40 ns twice. The first call uses the regular MagTense demagnetisation tensor, while the second enables the persistent dip-fmm plan. The normal MagTense `USE_CUDA` build flag and `cuda` problem setting also select dip-fmm's backend; the CPU path prefers oneMKL and falls back to portable execution. The core solver prints initialization and evaluation timings separately. Build MagTense with `USE_CDFMM=1` before running it.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

repository_root = Path.cwd()
while repository_root != repository_root.parent and not (repository_root / 'python' / 'src').is_dir():
    repository_root = repository_root.parent
python_source = repository_root / 'python' / 'src'
if str(python_source) not in sys.path:
    sys.path.insert(0, str(python_source))

from magtense.micromag import MicromagProblem

In [ ]:
GRID = (6, 6, 6)
CELL_SIZE = 5.0e-9
MS = 8.0e5
SIMULATION_TIME = 40.0e-9
OUTPUT_STEPS = 201
USE_CUDA = True  # Controls both regular MagTense and dip-fmm.

def make_problem(use_dip_fmm):
    m0 = np.zeros((int(np.prod(GRID)), 3))
    m0[:, 0] = 1.0
    m0[:, 2] = np.linspace(-0.1, 0.1, len(m0))
    m0 /= np.linalg.norm(m0, axis=1, keepdims=True)
    problem = MicromagProblem(
        res=list(GRID), grid_L=np.asarray(GRID) * CELL_SIZE,
        grid_type='uniform', solver='dynamic', m0=m0,
        A0=1.3e-11, Ms=MS, K0=0.0, alpha=4.42e3, gamma=0.0,
        tol=1.0e-3, cuda=USE_CUDA, cvode=False,
        usereturnhall=False,
        use_cdfmm=use_dip_fmm, cdfmm_order=6, cdfmm_depth=2,
        cdfmm_basis='spherical',
    )
    problem.window_enabled = 0
    problem.trace_enabled = 0
    return problem

def run_problem(use_dip_fmm):
    problem = make_problem(use_dip_fmm)
    result = problem.run_simulation(
        t_end=SIMULATION_TIME, nt=OUTPUT_STEPS,
        fct_h_ext=lambda times: np.zeros((len(times), 3)),
        nt_h_ext=2,
    )
    times = np.asarray(result[0])
    magnetisation = np.asarray(result[1][:, :, 0, :])
    return times, magnetisation

In [ ]:
# Regular MagTense demagnetisation call. Its timing is printed by Fortran.
print('=== Regular MagTense demag ===', flush=True)
regular_times, regular_magnetisation = run_problem(use_dip_fmm=False)

# Identical physical problem using dip-fmm. Its timing is printed separately.
print('=== dip-fmm demag ===', flush=True)
fmm_times, fmm_magnetisation = run_problem(use_dip_fmm=True)

relative_rms = (
    np.linalg.norm(fmm_magnetisation[-1] - regular_magnetisation[-1])
    / np.linalg.norm(regular_magnetisation[-1])
)
print(f'Simulated time: {SIMULATION_TIME * 1.0e9:.1f} ns')
print(f'MagTense cuda setting: {USE_CUDA}')
print(f'Final-state relative RMS difference: {relative_rms:.3e}')

In [ ]:
regular_mean = regular_magnetisation.mean(axis=1)
fmm_mean = fmm_magnetisation.mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(regular_times * 1.0e9, regular_mean[:, 0], label='Regular $m_x$')
ax.plot(fmm_times * 1.0e9, fmm_mean[:, 0], '--', label='dip-fmm $m_x$')
ax.set(xlabel='Time [ns]', ylabel='Mean normalized magnetisation')
ax.grid(alpha=0.3)
ax.legend()
plt.show()